# 🗄️ NOVA — GRPO Training Notebook

**Train Qwen2.5-1.5B to be a Senior DBA Agent using GRPO + LoRA**

Each episode: reset DB → agent picks index strategy → reward = did query cost drop?

> **Runtime:** GPU (T4 or better). `Runtime → Change runtime type → T4 GPU`  
> **Time:** ~20–40 minutes for 30 episodes

## 1. Clone Repository

In [ ]:
!git clone https://github.com/itsflash44/db-tune-project.git
%cd db-tune-project

## 2. Install Dependencies

In [ ]:
!pip install -q openenv-core>=0.1.13 trl>=0.8.0 peft>=0.10.0 \
    accelerate>=0.28.0 transformers>=4.40.0 datasets>=2.18.0 \
    fastapi uvicorn openai

## 3. Configuration

In [ ]:
import os

HF_TOKEN     = ''    # paste your HF token here
HF_REPO      = ''    # optional: 'yourname/nova-dba-lora'
NUM_EPISODES = 30

os.environ['HF_TOKEN']     = HF_TOKEN
os.environ['HF_REPO']      = HF_REPO
os.environ['NUM_EPISODES'] = str(NUM_EPISODES)
print(f'Episodes: {NUM_EPISODES}')

## 4. Direct Environment Test (No Server Needed)

In [ ]:
import sys
sys.path.insert(0, '/content/db-tune-project')

# Import the environment directly — no WebSocket, no server, no networking
from server.environment import DBEnvironment

# Test all three tasks
test_env = DBEnvironment()
for task in ['easy', 'medium', 'hard']:
    result = test_env.reset(task)
    print(f'Task={task} | cost={result["query_cost"]} | storage={result["storage_used"]}/{result["storage_budget"]} | indices={result["current_indices"]}')

print('\n✅ Environment works — no server or WebSocket needed!')

## 5. Direct Training Loop (GRPO-style Reward Tracking)

In [ ]:
import json, re, time, csv
import logging
from pathlib import Path
from openai import OpenAI
from server.environment import DBEnvironment
from reward_functions import StepState, reward_total, reward_cost_reduction, reward_storage_safety

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')

SYSTEM_PROMPT = """You are a Senior DBA Agent named NOVA. Your goal is to minimize query cost.
VALID COLUMNS (ONLY these are allowed for CREATE):
  department, salary, location, active_status
Output ONLY valid JSON, one of:
  {\"thought_process\": \"...\", \"command\": \"CREATE\", \"table_name\": \"users\", \"column_name\": \"<valid_column>\"}
  {\"thought_process\": \"...\", \"command\": \"DROP\",   \"table_name\": \"users\", \"column_name\": \"<index_name>\"}
  {\"thought_process\": \"...\", \"command\": \"FINISH\", \"table_name\": \"\",      \"column_name\": \"\"}
If cost <= 10, always FINISH."""

QUERY_MAP = {
    'easy':   'SELECT * FROM users WHERE department = \'Dept_5\'',
    'medium': 'SELECT * FROM users WHERE location = \'City_2\' AND active_status = 1',
    'hard':   'SELECT * FROM users WHERE department = \'Dept_9\'',
}

client_llm = OpenAI(
    api_key=HF_TOKEN,
    base_url='https://api-inference.huggingface.co/v1'
)
MODEL = 'Qwen/Qwen2.5-72B-Instruct'
MAX_TURNS = 10

OUTPUT_DIR = Path('outputs/nova-reward-run')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
reward_log = OUTPUT_DIR / 'reward_log.csv'

with open(reward_log, 'w', newline='') as f:
    csv.writer(f).writerow(['episode','task','total_reward','cost_reward','storage_reward','final_cost','timestamp'])

all_rewards = []

import itertools
tasks_cycle = itertools.cycle(['easy', 'medium', 'hard'])

def run_episode(ep_num):
    task = next(tasks_cycle)
    env = DBEnvironment()
    obs = env.reset(task)   # returns dict directly
    query = QUERY_MAP[task]
    conversation = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    total_r, cost_r_sum, storage_r_sum = 0.0, 0.0, 0.0
    prev_cost = float(obs['query_cost'])
    done = False

    for turn in range(MAX_TURNS):
        if done:
            break
        user_msg = (
            f'Task: {task}. Query: {query}. '
            f'Cost: {obs["query_cost"]}. '
            f'Storage: {obs["storage_used"]}/{obs["storage_budget"]}. '
            f'Indices: {obs["current_indices"]}. '
            'Next action?'
        )
        conversation.append({'role': 'user', 'content': user_msg})
        try:
            resp = client_llm.chat.completions.create(
                model=MODEL, messages=conversation,
                max_tokens=256, temperature=0.3
            )
            raw = resp.choices[0].message.content
        except Exception as e:
            logging.warning(f'LLM error: {e}'); break
        conversation.append({'role': 'assistant', 'content': raw})
        try:
            m = re.search(r'\{.*\}', raw, re.DOTALL)
            a = json.loads(m.group()) if m else {}
        except:
            a = {}
        action_dict = {
            'thought_process': a.get('thought_process', ''),
            'command': a.get('command', 'FINISH'),
            'table_name': a.get('table_name', 'users'),
            'column_name': a.get('column_name', ''),
        }
        step_result = env.step(action_dict)
        new_obs = step_result.get('observation', step_result)
        done = step_result.get('done', False)
        new_cost = float(new_obs.get('query_cost', prev_cost))
        state = StepState(
            prev_cost=prev_cost, new_cost=new_cost,
            storage_used=float(new_obs.get('storage_used', 0)),
            storage_budget=float(new_obs.get('storage_budget', 10)),
            command=action_dict['command'],
        )
        step_r = reward_total(state)
        cost_r_sum += reward_cost_reduction(state)
        storage_r_sum += reward_storage_safety(state)
        total_r += step_r
        prev_cost = new_cost
        obs = new_obs

    final_cost = float(obs.get('query_cost', prev_cost))
    all_rewards.append(total_r)
    mean10 = sum(all_rewards[-10:]) / len(all_rewards[-10:])
    print(f'Episode {ep_num:>3} | task={task} | reward={total_r:.2f} | cost={final_cost:.0f} | mean10={mean10:.2f}')
    with open(reward_log, 'a', newline='') as f:
        csv.writer(f).writerow([ep_num, task, total_r, cost_r_sum, storage_r_sum, final_cost, time.strftime('%Y-%m-%dT%H:%M:%S')])
    return total_r

print(f'Starting {NUM_EPISODES} episodes...')
for i in range(1, NUM_EPISODES + 1):
    run_episode(i)

print(f'\n✅ Done! Mean reward: {sum(all_rewards)/len(all_rewards):.2f}')

## 6. Reward Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_path = next(Path('outputs').glob('**/reward_log.csv'), None)
if log_path:
    df = pd.read_csv(log_path)

    # Rolling average
    df['rolling_reward'] = df['total_reward'].rolling(5, min_periods=1).mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('NOVA DBA Agent — GRPO Training Results', fontsize=14, fontweight='bold')

    for task, grp in df.groupby('task'):
        ax1.plot(grp['episode'], grp['total_reward'], alpha=0.4, label=f'{task} (raw)')
    ax1.plot(df['episode'], df['rolling_reward'], color='white', linewidth=2.5, label='Rolling avg (5)')
    ax1.axhline(y=1.5, color='lime', linestyle='--', linewidth=1.5, label='Target (+1.5)')
    ax1.set_facecolor('#1a1a2e'); fig.patch.set_facecolor('#1a1a2e')
    ax1.tick_params(colors='white'); ax1.title.set_color('white')
    ax1.set_title('Total Reward per Episode')
    ax1.set_xlabel('Episode', color='white'); ax1.set_ylabel('Reward', color='white')
    ax1.legend(fontsize=8)

    for task, grp in df.groupby('task'):
        ax2.plot(grp['episode'], grp['final_cost'], label=task, linewidth=2)
    ax2.axhline(y=10, color='lime', linestyle='--', linewidth=1.5, label='Target (cost=10)')
    ax2.set_facecolor('#1a1a2e')
    ax2.tick_params(colors='white'); ax2.title.set_color('white')
    ax2.set_title('Final Query Cost per Episode')
    ax2.set_xlabel('Episode', color='white'); ax2.set_ylabel('Cost', color='white')
    ax2.legend(fontsize=8)

    plt.tight_layout()
    out = OUTPUT_DIR / 'reward_curve.png'
    plt.savefig(str(out), dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print(f'✅ Saved to {out}')
    print('Download it from the left sidebar (Files icon)')
else:
    print('No reward log yet — run Cell 5 first.')